In [79]:
import sys

print(sys.executable)
print(sys.version)

/Users/abeerzafar/Desktop/AI-PITB-INTERNSHIP/.venv/bin/python
3.12.2 (v3.12.2:6abddd9f6a, Feb  6 2024, 17:02:06) [Clang 13.0.0 (clang-1300.0.29.30)]


In [80]:
import chromadb
from sentence_transformers import SentenceTransformer

print("ChromaDB:", chromadb.__version__)

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully")

ChromaDB: 1.5.9


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 7cf6988b-be50-4b85-890e-459092aff771)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Embedding model loaded successfully


In [81]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

print("API key loaded:", bool(api_key))

client = genai.Client(api_key=api_key)

API key loaded: True


In [83]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Explain RAG in two simple sentences."
)

print(response.text)

Retrieval-Augmented Generation (RAG) is an AI technique that looks up relevant facts from an external database before answering your question. This allows the AI to provide accurate, up-to-date responses instead of relying only on what it originally memorized.


In [84]:


response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Explain the complete RAG pipeline in 5 numbered steps, starting from uploading a PDF and ending with generating the final answer."
)

print(response.text)

Here is the complete Retrieval-Augmented Generation (RAG) pipeline explained in 5 sequential steps, from PDF ingestion to the final answer generation:

---

### **1. Document Parsing and Chunking (Ingestion)**
* **What happens:** You upload the PDF, and the pipeline extracts the raw text while removing irrelevant formatting. Because Large Language Models (LLMs) have context limits and perform better with focused information, the long document is broken down into smaller, manageable text segments called **chunks** (e.g., 500 words per chunk).
* **Key Detail:** An "overlap" (e.g., 50 words) is usually kept between adjacent chunks to ensure semantic context isn't lost at the boundaries where the text was split.

---

### **2. Vector Embedding and Indexing (Storage)**
* **What happens:** Each text chunk is passed through an **Embedding Model** (a specialized AI model) that converts the text into a dense numerical vector—a sequence of numbers that captures the semantic meaning of that chunk

## OBJECTIVE

The objective of this project is to build a Retrieval-Augmented Generation (RAG) based PDF assistant.The system allows users to upload a PDF document, extracts and chunks its text, converts the chunks into embeddings, storesthem in ChromaDB, and retrieves relevant information when the user asks a question.The retrieved information is then provided to llm to generate a source-grounded answer.

## ARCHITECTURE

1. PDF Upload

2. PDF Text Extraction
    
3. Text Cleaning
    
4. Text Chunking
    
5. SentenceTransformer Embeddings
    
6. ChromaDB
    
7. Semantic Retrieval
    
8. Relevant Context
    
9. LLM
    
10. Answer + Source

In [2]:
import os
import re
from pathlib import Path

import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from google import genai

print("Libraries imported successfully.")

Libraries imported successfully.


## CONFIGURATION

Loads environment variables and initializes the Gemini client using the API key from .env. This keeps credentials out of the notebook and centralizes model settings.

In [3]:
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in .env file")

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini client initialized.")

Gemini client initialized.


In [4]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
GEMINI_MODEL_NAME = "gemini-3.6-flash"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("LLM:", GEMINI_MODEL_NAME)

Embedding model: all-MiniLM-L6-v2
LLM: gemini-3.6-flash


## PDF EXTRACTION

Reads the uploaded PDF page by page using pypdf and pulls out raw text. This is the first ingestion step before any cleaning or chunking happens.

In [5]:
def extract_pdf_pages(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page_number, page in enumerate(
        reader.pages,
        start=1
    ):

        text = page.extract_text()

        if text and text.strip():

            pages.append({
                "page": page_number,
                "text": text.strip()
            })

    return pages

In [32]:
pdf_path = "Gen-AI.pdf"

pages = extract_pdf_pages(pdf_path)

print("Pages extracted:", len(pages))

Pages extracted: 5


In [33]:
for page in pages[:2]:

    print("\nPAGE:", page["page"])
    print(page["text"][:500])


PAGE: 1
Introduction to Generative AI Applications 
                               Practical Generative AI Use Cases 
 
Introduction to Generative AI:  
 
Generative AI refers to a category of artificial intelligence systems that can 
create new content  such as audio, video, text or other data rather than just 
analysing and classifying existing data. Generative AI models learn structure 
and patterns of large amount of data and use that understanding to produce 
new human like output. The most common 

PAGE: 2
Possible Improvement:  Connect the AI with the department's latest policy 
documents so it can always give correct and up-to-date information. 
Prompt: Write a formal reply email from the Ministry of Education to a 
student who is asking why their scholarship has not yet been granted, even 
though they applied a few months ago. Explain that the process takes time 
because the submitted documents must first be verified, and then the Ministry 
needs to receive confirmation from 

## CLEAN TEXT

Removes null characters and collapses extra whitespace from the extracted text. Clean input improves both embedding quality and chunk boundaries

In [63]:
def clean_text(text):

    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()

for page in pages:
    page["text"] = clean_text(page["text"])

## CHUNKING

Splits long page text into smaller overlapping segments so the LLM gets focused, context-limited pieces instead of a whole page at once. Overlap preserves meaning across chunk boundaries.

In [64]:
def chunk_text(
    text,
    chunk_size=800,
    overlap=150
):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

## PAGE CHUNKS

Applies the chunking function across every extracted page and assigns each chunk a unique ID, source, and page number for traceability.

In [65]:
chunks = []

for page in pages:

    page_chunks = chunk_text(
        page["text"]
    )

    for chunk_index, chunk in enumerate(page_chunks):

        chunks.append({
            "chunk_id": (
                f"page_{page['page']}_"
                f"chunk_{chunk_index}"
            ),
            "source": Path(pdf_path).name,
            "page": page["page"],
            "text": chunk
        })

print("Total chunks:", len(chunks))

Total chunks: 15


In [66]:
for chunk in chunks[:3]:

    print("Chunk:", chunk["chunk_id"])
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Text:", chunk["text"][:300])
    print("-" * 60)

Chunk: page_1_chunk_0
Source: Gen-AI.pdf
Page: 1
Text: Introduction to Generative AI Applications Practical Generative AI Use Cases Introduction to Generative AI: Generative AI refers to a category of artificial intelligence systems that can create new content such as audio, video, text or other data rather than just analysing and classifying existing d
------------------------------------------------------------
Chunk: page_1_chunk_1
Source: Gen-AI.pdf
Page: 1
Text: is given, an LLM generates a relevant response by predicting the most likely sequence of words. Use Case 1: AI Email Assistant Problem Statement: Government office staff spend alot of time by writing the same type of email repeatedly, like replying to citizens' complaints or asking for documents. Th
------------------------------------------------------------
Chunk: page_1_chunk_2
Source: Gen-AI.pdf
Page: 1
Text: d then writes a professional email in proper format. Expected Output: A ready-to-send professional email with c

## EMBEDDINGS

Converts each text chunk into a numerical vector using all-MiniLM-L6-v2, capturing its semantic meaning so it can later be compared against a query.

In [67]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=False
)

print("Number of chunks:", len(chunk_texts))
print("Embedding shape:", embeddings.shape)

Number of chunks: 15
Embedding shape: (15, 384)


## CHROMADB

Initializes a persistent ChromaDB collection to store chunk embeddings on disk, so the vector store survives across notebook restarts.

In [68]:
chroma_client = chromadb.PersistentClient(
    path="./chroma_db_notebook"
)

collection = chroma_client.get_or_create_collection(
    name="pdf_rag_assignment",
    metadata={
        "hnsw:space": "cosine"
    }
)

print("ChromaDB initialized successfully.")
print("Collection:", collection.name)
print("Existing documents:", collection.count())

ChromaDB initialized successfully.
Collection: pdf_rag_assignment
Existing documents: 0


## CHUNKS IN CHROMADB

Adds the chunk texts, embeddings, IDs, and metadata into the ChromaDB collection, making them searchable for retrieval.

In [90]:
ids = [
    chunk["chunk_id"]
    for chunk in chunks
]

metadatas = [
    {
        "source": chunk["source"],
        "page": chunk["page"],
        "chunk_id": chunk["chunk_id"]
    }
    for chunk in chunks
]

In [88]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(chunk_texts)

ids = [chunk["chunk_id"] for chunk in chunks]

metadatas = [
    {
        "source": chunk["source"],
        "page": chunk["page"],
        "chunk_id": chunk["chunk_id"]
    }
    for chunk in chunks
]

# CHECK LENGTHS
print("chunks:", len(chunks))
print("chunk_texts:", len(chunk_texts))
print("embeddings:", len(embeddings))
print("ids:", len(ids))
print("metadatas:", len(metadatas))


collection.add(
    ids=ids,
    documents=chunk_texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Documents stored:", collection.count())

chunks: 15
chunk_texts: 15
embeddings: 15
ids: 15
metadatas: 15
Documents stored: 15


In [70]:
collection.add(
    ids=ids,
    documents=chunk_texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print(
    "Documents stored:",
    collection.count()
)

Documents stored: 15


## SEMANTIC RETRIEVAL

Defines the function that embeds a user's query and retrieves the most semantically similar chunks from ChromaDB using cosine distance.

In [91]:
def retrieve_chunks(
    query,
    top_k=8
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )[0]

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    retrieved = []

    for i, document in enumerate(
        results["documents"][0]
    ):

        retrieved.append({
            "text": document,
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })

    return retrieved

## RETRIEVAL DEMO

The system first retrieves relevant document chunks before asking Gemini to generate the answer.

In [109]:
query = "What is the main purpose of this document?"

results = retrieve_chunks(query)

for i, result in enumerate(results, start=1):

    print(f"\nRESULT {i}")
    print("Source:", result["metadata"]["source"])
    print("Page:", result["metadata"]["page"])
    print("Distance:", result["distance"])
    print("Text:", result["text"][:500])


RESULT 1
Source: Gen-AI.pdf
Page: 4
Distance: 0.7053530812263489
Text: Target User: Citizens and government staff Input: A policy document and a specific question. AI Process: The AI searches the document and answers the question using only the information available in the document. Expected Output: A direct, accurate answer with a reference to where in the document it was found. Limitations: It may guess an answer if it is not present in the document and very large documents may be difficult to process. Possible Improvement: AI models tell clearly that this part i

RESULT 2
Source: Gen-AI.pdf
Page: 3
Distance: 0.7197657823562622
Text: Problem Statement: Public meetings, whether held in person or online (such as internal briefing) often go undocumented, making it hard to track decisions and follow ups later. Target User: Secretaries and official administrators Input: Input can be rough meeting notes, a written transcript, or a transcript generated from a recorded online meeting. AI Pr

## CONTEXT

Formats the retrieved chunks into a single structured context block (with source and page labels) that gets passed to Gemini as grounding material.

In [106]:
doc_overview = chunks[0]["text"]  # first chunk of the PDF — usually title/intro

def build_context(results, include_overview=True):
    context_parts = []
    if include_overview:
        context_parts.append(f"""
DOCUMENT OVERVIEW (opening section of the file)
File: {chunks[0]['source']}
Page: {chunks[0]['page']}
Content:
{doc_overview}
""")
    for i, result in enumerate(results, start=1):
        metadata = result["metadata"]
        context_parts.append(f"""
SOURCE {i}
File: {metadata['source']}
Page: {metadata['page']}
Content:
{result['text']}
""")
    return "\n\n".join(context_parts)

## GEMINI RAG PROMPT

Defines the system instruction and prompt template that force Gemini to answer strictly from the provided context, preventing hallucinated or outside-knowledge answers.

In [94]:
SYSTEM_INSTRUCTION = """
You are a PDF question-answering assistant.

Answer the user's question ONLY using the document
context provided to you.

Rules:

1. Do not use outside knowledge.
2. Do not invent information.
3. Do not make assumptions.
4. If the answer is not supported by the provided
   context, respond exactly:

"I could not find this information in the uploaded document."

5. Keep the answer clear and concise.
6. Do not mention information that is not present
   in the document.
"""


def generate_answer(
    question,
    context
):

    prompt = f"""
{SYSTEM_INSTRUCTION}

DOCUMENT CONTEXT:

{context}

USER QUESTION:

{question}
"""

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        contents=prompt
    )

    return response.text

## RETRIEVAL + GEMINI

Combines retrieval and generation into one end-to-end function: given a question, it retrieves context, builds the prompt, calls Gemini, and returns the answer with its sources.

In [107]:
def answer_question(question):

    results = retrieve_chunks(
        question,
        top_k=8
    )

    if not results:
        return {
            "answer": (
                "I could not find this information "
                "in the uploaded document."
            ),
            "sources": []
        }

    context = build_context(results)

    answer = generate_answer(
        question,
        context
    )

    sources = []

    for result in results:

        metadata = result["metadata"]

        sources.append({
            "source": metadata["source"],
            "page": metadata["page"]
        })

    return {
        "answer": answer,
        "sources": sources
    }

## TEST

In [108]:
result = answer_question(
    "What is the AI Email Assistant use case about?"
)

print("ANSWER:")
print(result["answer"])

print("\nSOURCES:")

for source in result["sources"]:

    print(
        f"- {source['source']} "
        f"(Page {source['page']})"
    )

ANSWER:
Based on the document context, the **AI Email Assistant** use case involves the following:

* **Problem Statement:** Government office staff spend a lot of time repeatedly writing the same type of emails (such as replying to citizens' complaints or requesting documents), which slows down work.
* **Target User:** Administrative staff across various government departments.
* **Input:** An email received from a citizen, along with key information to include in the reply (e.g., request receipts or processing timeframes).
* **AI Process:** The AI reads the citizen's email and the provided key information, then generates a professional email in the proper format.
* **Expected Output:** A ready-to-send professional email with a correct tone and no missing facts.
* **Limitations:** The AI might not know new policies unless provided, and it may reply too generally to legal or official matters, requiring a human to review it before sending.

SOURCES:
- Gen-AI.pdf (Page 1)
- Gen-AI.pdf (P